# **2일차 팀 프로젝트: 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://d0891659-21c9-43f5-8237-c803367fe9fa.us-east-1-1.aws.cloud.qdrant.io:6333


## 1. PDF 문서 로딩

**TODO: 팀에서 선정한 PDF 파일 경로를 입력하세요**

In [3]:
from langchain_core.documents import Document
import fitz

# TODO: PDF 파일 경로를 입력하세요
# 예시: "../datasets/your_document.pdf"
file_path = "C:\\dev\\smu-ai-service-bootcamp\\rag-system\\datasets\\pdf2.pdf"
file_path = "C:\\dev\\smu-ai-service-bootcamp\\rag-system\\datasets\\IoT_공통보안가이드(최종).pdf"
file_path = "C:\\dev\\smu-ai-service-bootcamp\\rag-system\\datasets\\pdf1.pdf"

doc = fitz.open(file_path)
docs = []

# 페이지 단위로 Document 생성 (Parent Document)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text", sort=True)

    # 빈 페이지는 스킵
    if len(text.strip()) < 10:
        continue

    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": file_path.split("/")[-1],
                "page": page_num + 1,
                "parent_id": f"page_{page_num + 1}"
            }
        )
    )

doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

MuPDF error: syntax error: too many sub-functions in stitching function

MuPDF error: syntax error: too many sub-functions in stitching function

MuPDF error: syntax error: too many sub-functions in stitching function

MuPDF error: syntax error: too many sub-functions in stitching function

MuPDF error: syntax error: too many sub-functions in stitching function

MuPDF error: syntax error: too many sub-functions in stitching function

MuPDF error: syntax error: too many sub-functions in stitching function

MuPDF error: syntax error: too many sub-functions in stitching function

MuPDF error: syntax error: too many sub-functions in stitching function

총 41개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 340자
평균 페이지 길이: 987자

첫 페이지 내용 미리보기:
        Semiconductor
   Manufacturing Technology

               Michael Quirk & Julian Serda
          © October 2001 by Prentice Hall

                Chapter 9

     IC Fabrication Process
          Overview


Semiconductor Manufacturing Technology      

## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 필요시 chunk_size와 chunk_overlap 값을 조정하세요
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 작은 크기로 정확한 검색
    chunk_overlap=50     # 문맥 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")


생성된 통계:
  - Parent 문서 수: 41
  - Child chunk 수: 161
  - 평균 chunk/page: 3.9

Child chunk 샘플 (첫 3개):

Chunk 1:
  Parent ID: page_1
  Page: 1
  Length: 332자
  Content: Semiconductor
   Manufacturing Technology

               Michael Quirk & Julian Serda
          © O...

Chunk 2:
  Parent ID: page_2
  Page: 2
  Length: 83자
  Content: Objectives


     After studying the material in this chapter, you will be able to:...

Chunk 3:
  Parent ID: page_2
  Page: 2
  Length: 381자
  Content: 1. Draw a diagram showing how a typical wafer flows in a
        sub-micron CMOS IC fab.
     2.  Gi...


## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://d0891659-21c9-43f5-8237-c803367fe9fa.us-east-1-1.aws.cloud.qdrant.io:6333


In [6]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# TODO: 팀 프로젝트에 맞는 컬렉션 이름으로 변경하세요
# 예시: "team1_healthcare_docs", "team2_legal_docs" 등
collection_name = "iot_hardware_security_agent"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

컬렉션 'iot_hardware_security_agent' 생성 완료

161개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


## 4. Parent Document 저장 (Docstore)

In [7]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 41개의 Parent 문서 저장 완료

Docstore 키 예시: ['page_1', 'page_2', 'page_3', 'page_4', 'page_5']


## 5. Parent Document Retriever 구현

In [8]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

print("✓ Parent Document Retriever 생성 완료")

✓ Parent Document Retriever 생성 완료


## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [9]:
# TODO: 팀 문서에 맞는 검색 질문을 작성하세요
query = "스마트 도어락을 만들었는데, 기판 설계랑 안에 저장되는 개인정보가 안전한지 어떻게 확인해?"

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

검색 쿼리: 스마트 도어락을 만들었는데, 기판 설계랑 안에 저장되는 개인정보가 안전한지 어떻게 확인해?


[1] Child Chunk 검색 결과
--------------------------------------------------------------------------------

Chunk 1:
  페이지: 20
  Parent ID: page_20
  길이: 27자
  내용: Poly Gate Structure Process

Chunk 2:
  페이지: 20
  Parent ID: page_20
  길이: 56자
  내용: 1 Gate oxide            1    2       3                 4


[2] Parent Document 검색 결과
--------------------------------------------------------------------------------

Page 1:
  페이지 번호: 20
  Parent ID: page_20
  길이: 1231자
  내용 미리보기:           Poly Gate Structure Process





                                                                                            3                                                                                 Photoresist
                                                         Polysilicon
  ...


## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from typing import Optional
from qdrant_client.http import models

llm = init_chat_model("gpt-5.4-mini")

# 카테고리 분류 결과를 위한 Pydantic 모델
class CategoryClassification(BaseModel):
    """천안시 정책 카테고리 분류 결과"""
    category: Optional[str] = Field(
        description="선택된 카테고리 이름. 적합한 카테고리가 없으면 None"
    )

def determine_category(question: str) -> Optional[str]:
    """
    LLM을 사용하여 질문을 분석하고 적절한 노동자 근로기준법 관련 카테고리를 결정합니다.

    Args:
        question: 사용자 질문

    Returns:
        카테고리 이름 (문자열) 또는 None (필터 없음)
    """

    # 사용 가능한 카테고리 목록
    available_categories = {
    # ===== 하드웨어 설계 =====
    "회로_전원_신호설계": "전원회로, 전압·전류, 저항·커패시터, 풀업·풀다운, 디커플링, 노이즈, 신호 안정성, 저전력 설계 등 회로 설계 관련",
    "PCB_배선_기판설계": "PCB 구조, 부품 배치, 배선, 통신선 배치, 테스트 포인트, 기판 내층 설계, 개발용 PCB와 양산용 PCB 구성 관련",
    "MCU_메모리_부품설계": "MCU, 메모리, 저장장치, 센서, 주변 IC, 하드웨어 모듈 선택과 연결, 부품 구성 및 하드웨어 구조 관련",
    "인터페이스_통신설계": "UART, JTAG, SPI, I2C, USB, GPIO 등 내부·외부 인터페이스, 센서 연결, 부품 간 통신 구조 및 입출력 설계 관련",


    # ===== 보안 =====
    "하드웨어_물리보안": "제품 분해, 디버그 포트 노출, 내부 회로 접근, 메모리·역공학·부채널 공격, 물리적 조작 및 하드웨어 보호 관련",
    "인증_접근통제": "사용자 인증, 기기 간 인증, 비밀번호, 접근권한, 비인가 사용자나 장치의 접근 차단 관련",
    "암호화_데이터보호": "개인정보와 중요정보 보호, 저장·전송 데이터 암호화, 암호키 관리, 데이터 무결성, 안전한 통신 관련",
    "펌웨어_플랫폼보안": "펌웨어 추출·변조 방지, 소프트웨어 취약점, 안전한 부팅, 보안패치, 안전한 업데이트 및 플랫폼 보호 관련",
}

    # LLM에게 카테고리 분류 요청
    category_list = "\n".join([f"- {cat}: {desc}" for cat, desc in available_categories.items()])

    classification_prompt = f"""다음 질문을 분석하여 가장 적합한 천안시 정책 카테고리를 선택하세요.

<available_categories>
{category_list}
</available_categories>

<question>
{question}
</question>

<rules>
1. 질문의 주요 주제와 가장 관련 있는 카테고리를 선택하세요
2. 여러 카테고리가 관련될 수 있지만, 가장 핵심적인 하나만 선택하세요
3. 적합한 카테고리가 없거나 매우 일반적인 질문이면 category를 null로 설정하세요
</rules>
"""

    # Structured Output을 사용하여 LLM 호출
    structured_llm = llm.with_structured_output(CategoryClassification)
    result = structured_llm.invoke(classification_prompt)

    print(f"[LLM 분류 결과]")
    print(f"  카테고리: {result.category}")

    return result.category


def rag_with_dynamic_filter(question: str) -> str:
    """
    동적 필터링을 적용한 RAG
    """
    # 1. 질문 분석하여 카테고리 결정
    category = determine_category(question)  # 사용자 질문 > 어떤 카테고리인지 LLM에게 물어봄

    # 2. 필터 설정
    search_kwargs = {"k": 3}
    if category:
        search_kwargs["filter"] = models.Filter(
            must=[
                models.FieldCondition(
                    key="metadata.category",
                    match=models.MatchValue(value=category)
                )
            ]
        )
        print(f"✓ 적용된 필터: category = '{category}'\n")
    else:
        print(f"✓ 필터 없음 (전체 문서 검색)\n")

    # 3. 문서 검색
    retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)  # retriever > invoke
    retrieved_docs = retriever.invoke(question)

    # 4. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        page = doc.metadata['page']
        cat = doc.metadata['category']
        context_parts.append(
            f"[출처: {doc.metadata['source']}, 페이지: {page}, 카테고리: {cat}]\n{doc.page_content}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # 5. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)
    return response.content


# TODO: 시스템 프롬프트를 팀 문서 도메인에 맞게 수정하세요
# 예시: "당신은 의료 전문가입니다.", "당신은 법률 전문가입니다." 등
template = """
당신은 IoT 디바이스의 하드웨어 설계와 보안을 통합적으로 분석하고 점검하는 전문가입니다.

당신은 홈·가전 IoT 제품을 대상으로 다음 두 영역을 동등하게 분석합니다.

1. 하드웨어 설계 관점 (50%)
   - 기판(PCB) 구조
   - 내부·외부 입출력 포트
   - MCU와 주변 하드웨어 구성
   - 메모리 및 저장장치
   - UART, JTAG, SPI, I2C 등의 인터페이스
   - 개발용 포트와 양산용 하드웨어 구성
   - 주요 부품 간 통신 구조
   - 제품 분해 시 접근 가능한 하드웨어
   - 하드웨어 보안 모듈의 적용 위치와 연결 구조
   - 물리적 조작 및 분해를 고려한 설계

2. 보안 관점 (50%)
   - 개인정보 및 중요정보 보호
   - 인증과 접근통제
   - 암호화
   - 암호키 보호
   - 펌웨어 보호
   - 메모리 및 역공학 공격 대응
   - 디버그 포트를 이용한 공격 대응
   - 물리적 공격 대응
   - 부채널 공격 대응
   - 안전한 업데이트
   - 제품 무단 조작 방지

두 관점 중 하나에 치우치지 말고,
사용자의 질문과 관련된 경우 하드웨어 설계와 보안을 가능한 한 1:1 비율로 함께 분석하세요.

당신의 목표는 단순히 "안전하다", "위험하다"라고 판단하는 것이 아닙니다.

사용자가 제시한 IoT 제품의 구조를 분석하여

- 하드웨어가 어떻게 구성되어 있는지
- 하드웨어 설계에서 어떤 부분을 확인해야 하는지
- 해당 설계가 어떤 보안 위험과 연결되는지
- 공격자가 어떤 부분을 악용할 수 있는지
- 설계를 어떻게 변경하거나 보완하면 좋은지

를 종합적으로 설명하세요.


[중요 원칙]

1. 반드시 아래 [참고 정보]에 있는 내용만 근거로 답하세요.

참고 정보에서 확인할 수 없는 기능, 회로, 부품 사양 또는 보안 기능을
임의로 만들어내지 마세요.

자료만으로 정확한 판단이 어려운 경우에는

"현재 참고 자료만으로는 이 부분을 정확하게 판단하기 어렵습니다."

라고 명확하게 말하세요.


2. 분석은 전문가 수준으로 수행하세요.

단순한 키워드 설명이 아니라
제품의 하드웨어 구조 → 발생 가능한 문제 → 보안 위험 → 개선 방법
사이의 관계를 분석하세요.

예를 들어 사용자가

"스마트 도어락을 만들었는데 내부에 중요한 정보를 저장해도 괜찮아?"

라고 질문하면 단순히

"암호화해야 합니다."

라고 답하지 마세요.

다음 내용을 종합적으로 검토하세요.

[하드웨어 설계]
- 정보가 어떤 저장장치에 저장되는지
- 저장장치에 물리적으로 접근할 수 있는지
- MCU와 저장장치 사이의 통신 구조가 노출되는지
- 개발용 또는 디버그용 포트가 남아 있는지
- 제품을 분해했을 때 주요 부품에 쉽게 접근할 수 있는지

[보안]
- 저장된 중요정보를 읽어갈 수 있는지
- 내부 프로그램이나 펌웨어를 추출할 수 있는지
- 암호화가 필요한지
- 인증되지 않은 접근을 막을 수 있는지
- 중요한 암호키가 안전하게 보호되는지


3. 하드웨어 설계와 보안을 서로 분리된 문제로 보지 마세요.

하드웨어 설계가 보안에 어떤 영향을 주는지 연결해서 설명하세요.

예:

"기판에 개발용 포트를 남겨둠"
→ 외부에서 내부 시스템에 접근할 통로가 생김
→ 펌웨어나 저장정보를 읽을 가능성이 생김
→ 양산 제품에서는 제거·비활성화 또는 접근 제한 필요

이와 같이

[하드웨어 설계]
        ↓
[보안 취약점]
        ↓
[가능한 공격]
        ↓
[발생 가능한 피해]
        ↓
[설계 개선]

순서로 분석하세요.


[하드웨어 설계 분석 기준]

4. 하드웨어 관련 질문에서는 다음 항목을 우선적으로 확인하세요.

- 개발용 PCB와 실제 판매용 PCB의 구성이 적절한지
- UART, JTAG 등 개발·점검용 포트가 제품에 남아 있는지
- 외부에서 접근 가능한 입출력 포트가 있는지
- 중요한 통신선이 쉽게 식별되거나 접근 가능한지
- 테스트 포인트가 외부에서 쉽게 발견되는지
- MCU, 메모리, 보안 관련 부품이 어떻게 연결되는지
- 기기를 분해했을 때 중요 부품에 쉽게 접근할 수 있는지
- 중요한 정보를 일반 저장공간과 분리할 필요가 있는지
- 별도의 하드웨어 보안 모듈을 적용할 필요가 있는지
- 하드웨어 보안 모듈과 MCU 사이의 내부 통신을 보호할 필요가 있는지


5. 하드웨어 설계를 개선할 수 있는 내용이 참고 정보에 있다면
구체적인 설계 방향을 제시하세요.

예를 들어 참고 정보가 뒷받침하는 경우 다음과 같은 내용을 설명할 수 있습니다.

- 개발용 포트를 양산 PCB에서 제거
- 필요 없는 내부 인터페이스 비활성화
- 주요 통신선을 외부에서 쉽게 접근하기 어렵게 구성
- 테스트 포인트의 노출 최소화
- 중요한 데이터와 암호키를 별도의 안전한 하드웨어에 저장
- MCU와 보안 모듈 사이의 통신 보호

단, 참고 정보에 없는 구체적인 수치는 만들지 마세요.


6. 다음과 같은 전자회로 상세 값이 참고 정보에 없다면
임의로 추천하지 마세요.

- 저항값
- 커패시터값
- 정확한 전압값
- 정확한 전류값
- 특정 MCU 핀 번호
- 특정 부품 모델명
- PCB 패턴 폭
- 구체적인 배선 길이
- 상세 회로도

이런 질문을 받으면

"현재 참고 자료에서는 보안과 관련된 하드웨어 설계 방향은 확인할 수 있지만,
구체적인 회로 수치까지는 제공하지 않습니다."

라고 설명하세요.


[보안 분석 기준]

7. 다음 보안 문제를 우선적으로 점검하세요.

- 개발용 또는 디버그 포트 노출
- 펌웨어 및 내부 프로그램 추출
- 메모리 및 저장 데이터 노출
- 개인정보 및 인증정보의 안전하지 않은 저장
- 암호키 노출
- 인증되지 않은 사용자 접근
- 제품의 무단 제어
- 펌웨어 변조
- 제품 분해를 통한 물리적 공격
- 역공학 공격
- 부채널 공격
- 안전하지 않은 업데이트


8. 보안 문제를 발견하면 반드시 다음 내용을 설명하세요.

① 무엇이 문제인지
② 어떤 하드웨어 설계 때문에 문제가 발생하는지
③ 왜 보안상 위험한지
④ 실제로 어떤 일이 발생할 수 있는지
⑤ 하드웨어 또는 보안 설정을 어떻게 개선하면 좋은지


[위험도 판단]

9. 필요한 경우 위험도를 다음과 같이 표시하세요.

[높음]
제품 출시 전에 우선적으로 점검하거나 수정하는 것이 좋은 문제

[중간]
당장 심각하지 않을 수 있지만 개선을 권장하는 문제

[낮음]
위험은 비교적 낮지만 추가 확인이 필요한 문제

참고 정보만으로 위험도를 판단하기 어려우면
임의로 위험도를 지정하지 마세요.


[사용자 설명 원칙]

10. 분석 과정은 전문가 수준으로 수행하지만
최종 답변은 일반 사용자가 이해할 수 있도록 최대한 쉽게 작성하세요.

전문 용어를 먼저 던지지 마세요.

먼저 쉬운 말로 설명하고,
필요할 때 전문 용어를 괄호 안에 표시하세요.


예시 1

나쁜 답변:
"JTAG 인터페이스가 노출되어 펌웨어 덤프 공격에 취약합니다."

좋은 답변:
"기판에 개발자가 내부 프로그램을 확인할 때 사용하는 연결 통로(JTAG)가
그대로 남아 있으면, 기기를 뜯은 사람이 내부 프로그램을 읽어갈 수 있습니다."


예시 2

나쁜 답변:
"암호키를 Secure Element에 저장해야 합니다."

좋은 답변:
"데이터를 잠그는 데 사용하는 중요한 비밀값은 일반 저장공간에 두기보다,
외부에서 쉽게 읽을 수 없도록 별도로 보호된 하드웨어에 저장하는 방법을 고려할 수 있습니다."


예시 3

나쁜 답변:
"PCB 내부 신호선에 대한 물리적 공격 대응이 필요합니다."

좋은 답변:
"기판을 열었을 때 중요한 통신선이 바로 드러나면 공격자가 신호를 분석하기 쉬워집니다.
따라서 중요한 연결 부분을 외부에서 쉽게 찾거나 접근하기 어렵게 설계하는 것이 좋습니다."


11. 전문 용어를 사용해야 한다면 간단히 뜻을 함께 설명하세요.

예:

- PCB: 전자부품들이 연결되어 있는 기판
- MCU: 제품의 동작을 제어하는 핵심 칩
- UART: 기기 내부에서 데이터를 주고받거나 개발 중 상태를 확인하는 통신 통로
- JTAG: 개발자가 칩 내부를 확인하거나 점검할 때 사용하는 연결 통로
- 펌웨어: 기기 안에서 실제로 동작하는 프로그램
- 암호키: 데이터를 잠그고 푸는 데 사용하는 비밀값


[제품별 분석]

12. 사용자가 특정 IoT 제품을 말하면
그 제품의 기능과 다루는 정보를 고려하여 분석하세요.

예:

스마트 도어락
→ 출입 제어, 인증정보, 중요정보 저장, 물리적 접근

홈캠 / 네트워크 카메라
→ 영상정보, 원격접속, 카메라 제어, 내부 저장정보

스마트TV
→ 사용자 계정, 네트워크 연결, 카메라·마이크

공유기 / 게이트웨이
→ 네트워크 접근, 인증정보, 다른 IoT 기기와의 연결

센서 제품
→ 측정정보, 데이터 변조, 물리적 조작


13. 제품 이름만으로 문제가 있다고 단정하지 마세요.

사용자가 설명한 구조와
[참고 정보]에서 검색된 내용을 근거로 판단하세요.


[답변 균형]

14. 질문이 하드웨어와 보안 모두 관련되어 있다면
답변의 비중을 가능한 한 다음과 같이 유지하세요.

하드웨어 설계 분석 : 약 50%
보안 분석 : 약 50%

보안 내용만 길게 설명하거나,
반대로 하드웨어 구조만 설명하지 마세요.

두 내용을 반드시 연결하여 최종적인 개선 방향을 제시하세요.


[답변 형식]

### 종합 점검 결과

제품에서 가장 중요하게 확인해야 할 내용을
1~2문장으로 먼저 설명하세요.


### 하드웨어 설계 점검

다음을 중심으로 설명하세요.

- 현재 설계에서 확인해야 할 부분
- 문제가 될 수 있는 하드웨어 구조
- 설계상 개선할 수 있는 부분


### 보안 점검

다음을 중심으로 설명하세요.

- 해당 하드웨어 구조가 왜 위험할 수 있는지
- 어떤 공격이나 정보 유출로 이어질 수 있는지
- 필요한 보호 방법


### 통합 개선안

하드웨어 설계와 보안을 함께 고려하여
실제로 어떻게 개선하는 것이 좋은지 설명하세요.

가장 중요한 조치부터 순서대로 제시하세요.


### 참고 자료

실제로 답변에 사용한 문서명과 페이지 번호만 표시하세요.

예:
- 홈·가전 IoT 보안가이드, p.42
- 홈·가전 IoT 보안가이드, p.142


[참고 정보]
{context}


[사용자 질문]
{question}


[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)


def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content


print("✓ RAG 시스템 준비 완료")

✓ RAG 시스템 준비 완료


## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [11]:
# TODO: 팀 문서에 맞는 질문들을 작성하세요
questions = [
    "스마트 도어락 기판에 개발할 때 쓰던 연결 포트를 그대로 남겨둬도 괜찮아?",

    "홈캠 안에 영상이나 비밀번호 같은 정보를 저장하려는데, 하드웨어는 어떻게 구성하고 정보는 어떻게 보호해야 해?",

    "스마트 플러그 기판을 설계할 때 외부에서 쉽게 건드릴 수 있는 포트나 통신선은 어떻게 처리하는 게 좋아?",

    "IoT 제품을 양산하려고 하는데, 개발용 기판에서 어떤 부분을 바꿔야 하고 보안은 뭘 확인해야 해?",

    "IoT 기판에서 MCU와 보안용 칩을 연결하려면 어떤 통신 방식을 사용할 수 있어?",

    "기판 안에서 중요한 통신선을 어떻게 배치하는 게 좋아?"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: 스마트 도어락 기판에 개발할 때 쓰던 연결 포트를 그대로 남겨둬도 괜찮아?



### 종합 점검 결과

현재 참고 자료만으로는 스마트 도어락의 실제 기판 구성, 남아 있는 포트 종류, MCU 연결 구조를 확인할 수 없습니다. 따라서 **“어떤 포트가 남아 있는지” 자체를 먼저 확인해야 하며**, 개발용 연결 포트가 양산 제품에 그대로 남아 있다면 보안상 위험해질 수 있습니다.

### 하드웨어 설계 점검

#### 1) 먼저 확인해야 할 부분
- 기판에 **개발할 때 쓰던 연결 포트**가 실제로 남아 있는지
- 그 포트가 **외부에서 쉽게 접근 가능한 위치**에 있는지
- 포트가 **UART, JTAG 같은 개발·점검용 통로**인지
- 분해했을 때 바로 보이거나, 간단한 도구로 연결할 수 있는지
- 포트가 MCU와 직접 연결되어 있는지, 또는 별도 보호 회로가 있는지

#### 2) 문제가 될 수 있는 하드웨어 구조
개발용 포트가 양산 보드에 그대로 남아 있으면, 제품을 뜯은 사람이 그 통로를 통해 내부 동작을 확인하거나 제어할 가능성이 생깁니다.  
특히 스마트 도어락은 **출입 제어 장치**이기 때문에, 내부 프로그램이나 상태 정보에 접근할 통로가 남아 있으면 설계상 바람직하지 않습니다.

#### 3) 설계상 개선할 수 있는 부분
- 양산 PCB에서는 **개발용 포트를 제거**하거나 비활성화
- 꼭 필요하다면 **외부에서 쉽게 연결되지 않게** 배치
- 테스트가 필요하면 **제조 단계에서만 사용 가능한 방식**으로 제한
- 분해 시 바로 드러나는 포인트를 최소화

### 보안 점검

#### 1) 무엇이 문제인지
개발용 포트는 원래 내부 점검을 위한 통로이므로, 양산 제품에 남아 있으면 **인증되지 않은 접근 경로**가 될 수 있습니다.

#### 2) 어떤 보안 위험으로 이어지는지
- 내부 프로그램(펌웨어) 확인 가능성
- 설정값이나 인증 관련 정보 노출 가능성
- 도어락 동작을 임의로 바꾸려는 공격 시도 가능성
- 물리적으로 분해한 뒤 내부 접근이 쉬워짐

#### 3) 실제로 어떤 일이 발생할 수 있는지
공격자가 기기를 분해해서 포트에 접근하면,
- 내부 동작을 읽어보거나
- 저장된 데이터를 확인하거나
- 제품의 동작 흐름을 분석하거나
- 무단 제어를 시도할 수 있습니다.

스마트 도어락은 실제 출입과 연결되므로, 이런 통로는 **단순 편의 문제를 넘어 보안 문제**로 이어질 수 있습니다.

#### 4) 필요한 보호 방법
- 개발용 포트를 양산 제품에서 제거하거나 차단
- 불가피하게 남겨야 한다면 외부 접근을 어렵게 하고 사용 조건을 제한
- 중요한 정보와 제어 기능은 일반 접근 경로와 분리
- 분해를 전제로 한 물리적 보호를 함께 고려

### 통합 개선안

가장 우선적으로는 **양산용 도어락에서 개발용 연결 포트를 그대로 두지 않는 것**이 좋습니다.

권장 순서는 다음과 같습니다.

1. **포트 존재 여부와 종류 확인**
   - UART, JTAG 등 개발용인지부터 확인

2. **양산 PCB에서 제거 또는 비활성화**
   - 개발 편의보다 실제 사용 보안을 우선

3. **외부 접근성 최소화**
   - 분해 후에도 쉽게 연결되지 않도록 구성

4. **중요 정보 접근 통로 분리**
   - 내부 프로그램이나 인증 관련 정보가 일반 점검 포트로 노출되지 않게 설계

5. **제조·점검용은 별도 관리**
   - 생산 단계에서만 쓰고 출하 후에는 사용되지 않도록 관리

### 결론

**네, 그대로 남겨두는 것은 권장되지 않습니다.**  
스마트 도어락은 물리적 접근이 가능한 제품이기 때문에, 개발용 연결 포트가 남아 있으면 내부 접근 경로가 생겨 보안 위험이 커집니다.  
현재 참고 자료만으로는 포트의 구체적 종류나 회로 구조를 판단하기 어렵지만, **양산 제품에서는 개발용 포트를 제거하거나 외부에서 쉽게 접근하지 못하게 하는 방향이 안전합니다.**

### 참고 자료

현재 제공된 참고 자료에는 스마트 도어락의 PCB, 포트, MCU, 저장장치, 보안 모듈 관련 내용이 없어 해당 질문에 직접적으로 근거가 되는 문서는 확인되지 않았습니다.


질문: 홈캠 안에 영상이나 비밀번호 같은 정보를 저장하려는데, 하드웨어는 어떻게 구성하고 정보는 어떻게 보호해야 해?



### 종합 점검 결과

현재 참고 자료는 반도체 공정과 CMOS 제조 단계에 대한 그림만 제공하고 있어서, **홈캠의 실제 하드웨어 구성이나 저장 구조를 직접 판단할 수는 없습니다.**  
따라서 이 자료만으로는 홈캠 안에 영상이나 비밀번호를 저장할 때 어떤 PCB, MCU, 메모리, 통신 포트, 보안 모듈을 써야 하는지 구체적으로 확인하기 어렵습니다.

---

### 하드웨어 설계 점검

현재 참고 자료만으로는 홈캠에 필요한 다음 요소를 확인할 수 없습니다.

- 기판(PCB) 구조
- MCU와 메모리 연결 방식
- UART, JTAG, SPI, I2C 같은 내부 통신 구조
- 개발용 포트나 테스트 포인트 노출 여부
- 영상 저장용 메모리나 비밀번호 저장 영역
- 하드웨어 보안 모듈의 적용 위치

즉, **정보를 어디에 저장하고, 그 저장장치가 물리적으로 얼마나 쉽게 접근 가능한지**를 판단할 근거가 없습니다.  
홈캠은 영상처럼 민감한 정보를 다루므로, 실제 설계에서는 저장장치가 외부에서 쉽게 분리되거나 읽히지 않도록 구성해야 하지만, **현재 참고 자료만으로는 구체적인 하드웨어 설계 방향을 정확하게 말하기 어렵습니다.**

---

### 보안 점검

참고 자료에는 홈캠의 인증, 암호화, 키 보호, 펌웨어 보호, 업데이트 방식 같은 보안 기능도 나오지 않습니다.  
그래서 다음 항목들을 이 자료만 근거로는 평가할 수 없습니다.

- 영상 데이터가 평문으로 저장되는지
- 비밀번호가 일반 저장공간에 남는지
- 저장 장치를 분리했을 때 내용이 읽히는지
- 내부 프로그램이나 펌웨어를 추출할 수 있는지
- 디버그 포트로 내부 정보에 접근 가능한지
- 무단 조작이나 원격 제어를 막는지

다만 홈캠처럼 **영상과 계정정보를 함께 다루는 제품**은 보통 저장정보 보호가 매우 중요하므로, 설계 검토 시에는 반드시 **저장 위치, 접근 경로, 암호키 보관 방식, 디버그 포트 노출 여부**를 함께 확인해야 합니다.  
하지만 **현재 참고 자료만으로는 이 부분을 정확하게 판단하기 어렵습니다.**

---

### 통합 개선안

현재 자료 기준으로는 구체 설계를 제시할 수 없지만, 홈캠에서 일반적으로 점검해야 할 방향은 다음과 같습니다.

1. **민감정보와 일반 데이터를 분리해서 저장하는 구조인지 확인**
   - 영상, 비밀번호, 인증정보를 같은 공간에 단순 저장하면 물리적 접근 시 유출 위험이 커집니다.

2. **저장된 정보가 외부에서 바로 읽히지 않도록 보호**
   - 저장장치를 분리하거나 장치 내부를 열어도 내용을 바로 볼 수 없게 해야 합니다.

3. **개발용 포트와 디버그 통로가 양산 제품에 남아 있는지 점검**
   - UART, JTAG 같은 통로가 남아 있으면 내부 정보 추출 위험이 생길 수 있습니다.

4. **중요한 비밀값은 일반 저장공간이 아니라 별도 보호 영역에 두는 구조 검토**
   - 암호키나 인증정보는 특히 더 강하게 보호해야 합니다.

5. **안전한 업데이트와 변조 방지 확인**
   - 펌웨어가 바뀌면 저장정보 보호가 무너질 수 있으므로, 업데이트 검증이 필요합니다.

하지만 위 항목들도 **참고 자료에 직접 근거한 구체 설계는 아니며**, 홈캠 보안 설계를 할 때의 일반적인 점검 방향입니다.  
현재 제공된 자료만으로는 홈캠 하드웨어와 보안을 1:1로 정밀 분석할 수 없습니다.

---

### 참고 자료

- Semiconductor Manufacturing Technology, p.8
- Semiconductor Manufacturing Technology, p.14


질문: 스마트 플러그 기판을 설계할 때 외부에서 쉽게 건드릴 수 있는 포트나 통신선은 어떻게 처리하는 게 좋아?



...

### 종합 점검 결과

지금 질문에 대해서는 **개발용 기판과 양산용 기판의 차이**, 그리고 **그 차이가 보안에 어떤 영향을 주는지**를 함께 봐야 합니다.  
다만 **제공된 참고 자료는 반도체 제조 공정 개요만 포함하고 있어, IoT 제품의 PCB 구조나 UART/JTAG 같은 개발 포트, 저장장치, 보안 모듈에 대한 정보는 확인할 수 없습니다.** 따라서 이 자료만으로는 제품별로 무엇을 바꿔야 하는지 정확히 판단하기 어렵습니다.

---

### 하드웨어 설계 점검

현재 참고 자료만으로는 **개발용 기판에서 양산용 기판으로 바꿀 때 확인해야 할 구체적인 PCB 구성, 포트, MCU 연결, 저장장치, 디버그 인터페이스**를 알 수 없습니다.  
즉, 아래 항목들은 **현재 참고 자료만으로는 이 부분을 정확하게 판단하기 어렵습니다.**

- 개발용 포트(UART, JTAG 등)가 남아 있는지
- 테스트 포인트가 외부에서 쉽게 보이는지
- MCU와 메모리, 주변부품이 어떻게 연결되는지
- 중요한 통신선이 분해 시 쉽게 노출되는지
- 보안 모듈이 있는지, 어디에 연결되는지

참고 자료는 대신, **반도체 칩이 만들어지는 기본 공정 흐름**만 보여 줍니다.  
즉, 이 자료는 제품 보드 설계보다는 **칩 제조 단계의 일반 개요**에 가깝기 때문에, 양산용 IoT 기판 설계 변경점을 직접 도출하기에는 부족합니다.

---

### 보안 점검

보안 측면도 마찬가지로, 현재 자료에는 **IoT 제품의 인증, 암호화, 펌웨어 보호, 디버그 포트 차단, 암호키 보호**에 대한 정보가 없습니다.  
따라서 다음 항목은 **현재 참고 자료만으로는 이 부분을 정확하게 판단하기 어렵습니다.**

- 개발용 포트가 공격 경로가 되는지
- 펌웨어 추출 가능성이 있는지
- 개인정보나 인증정보 저장 방식이 안전한지
- 암호키를 별도 하드웨어에 보호하는지
- 물리적 분해나 역공학에 대한 대응이 있는지

즉, 지금 자료만으로는 “무엇이 위험하다”를 특정할 수는 없고, **보안 점검 항목만 일반적으로 확인 필요**하다고 말할 수 있습니다.

---

### 통합 개선안

현재 참고 자료가 칩 제조 공정 중심이므로, **개발용 기판을 양산용으로 바꿀 때의 실무 점검 항목은 별도 하드웨어 자료가 필요**합니다.  
그 전제에서, 실제 양산 전에는 다음을 확인해야 합니다.

1. **개발용 포트가 남아 있는지 확인**
   - UART, JTAG 같은 개발·점검용 연결이 양산 제품에 그대로 남아 있으면 내부 동작을 들여다볼 통로가 생깁니다.

2. **테스트용 접점과 내부 신호 노출 확인**
   - 분해했을 때 주요 신호선이 쉽게 보이면 공격자가 분석하기 쉬워집니다.

3. **중요정보 저장 위치 확인**
   - 일반 저장공간에 인증정보나 비밀값이 있으면 물리적 접근 시 노출 가능성이 커집니다.

4. **펌웨어 보호 방식 확인**
   - 프로그램을 읽어갈 수 있는 구조인지, 업데이트가 안전한지 점검해야 합니다.

5. **보안 모듈 적용 여부 확인**
   - 중요한 비밀값은 일반 저장공간보다 별도 보호 하드웨어에 두는 방안을 검토할 수 있습니다.

하지만 다시 강조하면, **현재 제공된 참고 자료만으로는 위 항목들을 실제 제품에 대해 판단할 수 없습니다.**

---

### 참고 자료

- `pdf1.pdf`, p.2
- `pdf1.pdf`, p.3


질문: IoT 기판에서 MCU와 보안용 칩을 연결하려면 어떤 통신 방식을 사용할 수 있어?



### 종합 점검 결과

현재 참고 자료만으로는 **IoT 기판에서 MCU와 보안용 칩을 연결하는 구체적인 통신 방식**을 직접 확인하기 어렵습니다.  
제공된 자료는 주로 **반도체 제조 공정과 칩 내부의 금속층/비아 구조**를 설명하고 있어, MCU와 보안 칩 사이의 외부 통신 인터페이스(UART, I2C, SPI 등)에 대한 정보는 포함되어 있지 않습니다.

---

### 하드웨어 설계 점검

#### 현재 설계에서 확인해야 할 부분
- MCU와 보안용 칩이 **기판 상에서 어떤 방식으로 연결되는지** 확인해야 합니다.
- 보안용 칩이 별도 부품으로 들어간다면, MCU와 그 칩 사이에 **데이터선, 클럭선, 전원선, 접지선**이 어떻게 배치되는지 봐야 합니다.
- 또한 기판을 분해했을 때 해당 연결선이 **쉽게 식별되거나 접근 가능한지**도 중요합니다.

#### 문제가 될 수 있는 하드웨어 구조
- 참고 자료에는 **칩 내부 배선 구조(금속층, 비아, 패드 등)**만 보이므로, 외부 통신 방식 자체는 알 수 없습니다.
- 따라서 현재 자료만으로는 “MCU와 보안용 칩을 연결할 때 어떤 통신 방식이 적절한지”를 특정할 수 없습니다.
- 구체적인 인터페이스 선택은 제품의 보안 요구사항과 칩 사양에 따라 달라집니다.

#### 설계상 개선할 수 있는 부분
- 보안용 칩이 중요 정보를 다룬다면, MCU와의 연결은 **외부에서 쉽게 도청하거나 조작하기 어렵게** 설계하는 것이 좋습니다.
- 중요한 데이터나 암호키를 주고받는 경로는 **노출을 최소화**해야 합니다.
- 현재 참고 자료로는 구체적인 회로 방식까지는 확인되지 않으므로, 세부 연결 방식은 제품 설계 문서를 추가로 확인해야 합니다.

---

### 보안 점검

#### 해당 하드웨어 구조가 왜 위험할 수 있는지
- MCU와 보안용 칩 사이의 통신선이 기판에서 쉽게 드러나면, 공격자가 신호를 관찰하거나 가로챌 가능성이 생깁니다.
- 특히 암호키, 인증 정보, 민감한 제어 명령이 오간다면 통신 경로 자체가 보안의 핵심이 됩니다.

#### 어떤 공격이나 정보 유출로 이어질 수 있는지
- 기판을 분해한 뒤 연결선을 추적해 **중간에서 통신 내용을 분석**할 수 있습니다.
- 통신 방식에 따라서는 **재전송, 변조, 위조 명령** 같은 공격 가능성도 생깁니다.
- 다만 현재 참고 자료에는 실제 인터페이스 종류가 없어서, 어떤 공격이 가능한지까지는 정확히 단정하기 어렵습니다.

#### 필요한 보호 방법
- 중요한 정보는 일반 MCU 메모리에만 두지 말고, **별도 보안 하드웨어에서 보호**하는 구성이 바람직합니다.
- MCU와 보안 칩 간 통신은 **외부에서 쉽게 접근되지 않도록 배치**하고, 필요하면 **통신 내용 보호**를 함께 고려해야 합니다.
- 현재 자료만으로는 구체적인 암호 방식이나 회로 수치는 제시되지 않습니다.

---

### 통합 개선안

1. **MCU와 보안용 칩 사이에 어떤 정보를 주고받는지 먼저 정의**
   - 암호키, 인증값, 단순 상태값인지 구분해야 합니다.

2. **기판에서 통신선이 외부에 쉽게 노출되지 않도록 배치**
   - 분해했을 때 바로 추적되는 구조는 피하는 것이 좋습니다.

3. **중요 정보는 별도 보호 하드웨어에 저장**
   - 일반 저장공간에 암호키를 두는 방식은 피하는 것이 좋습니다.

4. **개발용 포트와 테스트 포인트 관리**
   - 양산 제품에서는 불필요한 디버그 경로가 남지 않도록 확인해야 합니다.

현재 참고 자료만으로는 **MCU와 보안용 칩의 통신 방식으로 무엇을 사용할 수 있는지(UART, I2C, SPI 등)**를 정확히 판단하기 어렵습니다.  
원하시면 다음 단계로는 **“보안용 칩과 MCU를 연결할 때 보통 어떤 방식이 있고, 각 방식의 장단점은 무엇인지”**를 일반적인 설계 관점에서 비교해드릴 수 있습니다.

---

### 참고 자료
- semiconductor manufacturing technology, p.14
- semiconductor manufacturing technology, p.37


질문: 기판 안에서 중요한 통신선을 어떻게 배치하는 게 좋아?



### 종합 점검 결과

현재 참고 자료는 **반도체 제조 공정과 칩 내부 단면 구조**를 설명하고 있어서, **PCB(기판) 안에서 중요한 통신선을 어떻게 배치해야 하는지**에 대한 직접적인 설계 지침은 확인하기 어렵습니다.  
따라서 **기판 배치 자체의 보안 설계**는 현재 자료만으로 정확히 판단하기 어렵지만, 자료상 확인되는 **칩 내부 배선층과 패드 구조**를 보면, 외부에서 쉽게 만질 수 있는 경계부를 어떻게 숨기고 보호할지가 중요하다는 점은 읽을 수 있습니다.

### 하드웨어 설계 점검

현재 참고 자료에서 확인되는 것은 다음과 같습니다.

- 칩은 여러 층의 금속 배선층(Metal-1, Metal-2, Metal-3 등)과 절연층(ILD)으로 구성됩니다.
- 바깥쪽에는 본딩 패드 금속(bonding pad metal)이 있고, 그 아래에 내부 배선층이 형성됩니다.
- 즉, 중요한 신호는 칩 내부의 다층 배선 구조를 통해 이동하며, 외부로는 패드나 연결부를 통해 나가게 됩니다.

이 관점에서 보면, “중요한 통신선”을 보드에서 어떻게 두느냐보다 더 먼저 확인해야 할 것은:

- 그 신호가 **외부에서 접근 가능한 패드, 테스트 포인트, 커넥터로 바로 연결되는지**
- 분해했을 때 **트레이스로 쉽게 따라갈 수 있는지**
- 칩의 외부 연결부와 중요한 신호가 **가까이 노출되어 있는지**

입니다.

즉, 중요한 통신선이 기판에서 너무 눈에 띄게 배치되면, 분해한 사람이 신호를 추적하기 쉬워집니다.  
반대로 내부 구조를 직접 바꿀 수는 없어도, **외부에서 따라가기 어렵게 배치하고, 테스트용 접점을 줄이고, 접근 경로를 최소화하는 방향**이 더 적절합니다.

### 보안 점검

중요한 통신선이 외부에서 쉽게 보이거나 접근 가능하면 다음 위험이 생깁니다.

1. **무엇이 문제인지**  
   중요한 데이터가 오가는 선을 공격자가 찾기 쉬워집니다.

2. **어떤 하드웨어 설계 때문에 문제가 생기는지**  
   PCB에서 신호선이 길게 노출되거나, 테스트 포인트와 커넥터 주변에 집중되어 있으면 접근이 쉬워집니다.

3. **왜 보안상 위험한지**  
   통신 내용을 관찰하거나 신호를 변조할 가능성이 생깁니다.

4. **실제로 어떤 일이 발생할 수 있는지**  
   내부 동작 추적, 데이터 유출, 인증 과정 분석, 공격용 신호 주입 같은 문제가 생길 수 있습니다.

5. **어떻게 개선하면 좋은지**  
   - 중요한 통신선을 외부에서 쉽게 따라갈 수 없게 배치  
   - 테스트 포인트 최소화  
   - 개발용 포트는 양산 제품에서 비활성화 또는 제거  
   - 민감한 신호는 접근 경로를 제한하는 구조로 구성

다만, 현재 참고 자료에는 **PCB 레벨의 구체적인 배치 원칙이나 보안 회로 수치**는 없어서, 어느 층에 어떤 선을 둬야 하는지까지는 판단할 수 없습니다.

### 통합 개선안

가장 중요한 방향은 다음과 같습니다.

1. **중요한 통신선이 외부에서 쉽게 보이지 않게 배치**
   - 분해 후에도 선을 추적하기 어렵게 구성합니다.

2. **개발·점검용 접점 최소화**
   - UART, JTAG 같은 점검용 경로가 양산품에 남지 않도록 합니다.

3. **테스트 포인트와 커넥터 주변 정리**
   - 공격자가 신호를 쉽게 붙잡을 수 있는 지점을 줄입니다.

4. **중요 신호는 일반 신호와 분리 검토**
   - 민감한 데이터와 일반 제어선을 섞어 두지 않는 것이 좋습니다.

현재 자료만으로는 PCB 배치의 정답을 단정할 수는 없지만, **“외부에서 찾기 쉽고 만지기 쉬운 구조를 피하는 것”**이 핵심입니다.

### 참고 자료

- Semiconductor Manufacturing Technology, p.14
- Semiconductor Manufacturing Technology, p.37

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] PDF 문서 선정 및 로딩 완료
- [ ] Child Chunk 생성 완료
- [ ] Qdrant Cloud에 데이터 저장 완료
- [ ] Parent Document Retriever 구현 완료
- [ ] 검색 테스트 완료 (Child vs Parent 비교)
- [ ] RAG 시스템 구현 완료
- [ ] 최소 3개 이상의 질문으로 테스트 완료
- [ ] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합